## Goal and roadmap
- Goal (plain): Turn your team's coding rules and agent boundaries into two clear docs that AI coding assistants can follow, and ship a small validator so you can check them in seconds before a PR lands — all locally in this notebook.

- What this notebook does:
  1) Writes a repository-level guide for assistants (.github/copilot-instructions.md). 2) Writes AGENTS.md that defines named personas, allowed actions, templates, and boundaries. 3) Generates a tiny Python validator (tools/validate_agent_docs.py). 4) Runs the validator and iterates until it passes. 5) Commits the files locally (no push unless you do it later).

- What you should be able to do after: Explain why repo-level instructions matter, map your Python repo conventions into precise rules and examples, define agent roles with clear boundaries, and run/automate a validation check (locally and in CI) so gaps are caught early.

- Assumes you know: basic Python, Markdown, and git basics (clone/commit). Likely gap to slow down for: how Python repos are typically laid out (src/, package root, tests/), what a repo-level assistant instruction file controls, how to structure agent personas, and how to do minimal programmatic Markdown checks. We teach these inline.

Note: This is an artifact-building lesson. Cells will write files to your current repository and validate them. No network calls or remote pushes run here.

### Pipeline map — what we build and check
The boxes are the concrete files and checks. Keep this handy as the reference for the rest of the notebook.

```text
[Check environment]
       |
       v
[Write .github/copilot-instructions.md]
       |
       v
[Quick local check: headings/examples/tokens]
       |
       v
[Write AGENTS.md]
       |
       v
[Quick local check: personas/sections/templates]
       |
       v
[Write tools/validate_agent_docs.py]
       |
       v
[Run validator -> JSON + summary]
       |
       v
[If pass] --> [git add/commit (local only)]
   |
[If fail] -- print issues -> edit docs -> re-run
```
Boxes labeled "Write …" create real files. Checks are runnable validations that must pass before committing.

### Why repo-level assistant instructions exist (quick intuition)
- What it is: A single, versioned Markdown file that spells out coding style, layout, tests, and safety rails for AI coding assistants. Think of it as a project-wide prompt that every assistant reads before suggesting changes.
- Why it matters: Vague prompts produce risky or off-style code. Precise, repo-aware rules turn suggestions into consistent, testable diffs.

Ambiguous vs precise:
- Ambiguous: "Add logging to functions."
- Precise (repo-level): "Add structured logging using stdlib logging; do not introduce new deps; place config in src/my_package/logging.py; include tests in tests/test_logging.py; run pytest -q locally before proposing changes."

### Primer — Python repo layout to codify
- src/ layout: Put library code under src/<package_name>/ so imports are explicit (PYTHONPATH includes src/ during dev). Example: src/my_package/module.py.
- tests/: Put tests under tests/, mirroring the package structure. Run with pytest.
- Typing and lint: Require type hints and a static check (mypy). Keep lint/format consistent (e.g., ruff/black/isort).
- Why agents need this: Without it, they guess paths/imports, which breaks imports and CI.

In [ ]:
# Setup check — run me first
import importlib, sys, shutil, subprocess, os, json, re
from pathlib import Path

# Only standard library is used.
REQUIRED = {}  # stdlib only
missing = [f"{m} ({hint})" for m, hint in REQUIRED.items()
           if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit("Missing prerequisites:\n  - " + "\n  - ".join(missing))

# Detect git and whether we're in a repo (for the commit step later)
GIT = shutil.which("git")
repo_root = Path.cwd()
in_git_repo = (repo_root / ".git").exists()
branch = None
if GIT and in_git_repo:
    try:
        branch = subprocess.check_output([GIT, "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
    except Exception:
        branch = None

print("Setup OK — Python", sys.version.split()[0])
print("git on PATH:", bool(GIT))
print("In git repo:", in_git_repo, (f"branch={branch}" if branch else ""))
print("Working dir:", repo_root)

You should see your Python version, whether git is available, and whether the current directory is a git repo. The commit step will automatically skip if either git is missing or you are not inside a repo.

### Write .github/copilot-instructions.md — structure we expect
Required sections (case-insensitive headings):
- Purpose — why this file exists for assistants.
- Scope — what assistants may/should do vs not do.
- Coding conventions — concrete repo norms: src/ layout, imports, typing, tests, lint commands.
- Examples — at least one Before/After with code blocks.
- Security constraints — forbidden actions/tokens; how to handle secrets and internal endpoints.
- Escalation — when to stop and ask a human.

We also require a mention of at least one repo token: "src/", "tests/", "pytest", or "mypy".

In [ ]:
# Artifact: write .github/copilot-instructions.md
from textwrap import dedent

instructions_path = repo_root / ".github" / "copilot-instructions.md"
instructions_path.parent.mkdir(parents=True, exist_ok=True)

instructions_md = r"""
# Copilot repository instructions

## Purpose
Guide AI coding assistants and code-suggestion tools so generated changes follow this repository's conventions, pass tests, and respect security boundaries.

## Scope
These instructions apply to all automated or AI-assisted changes proposed in this repository. Assistants may create or modify code, refactor modules, and write tests. Assistants must not introduce new external dependencies without an explicit request and justification. Prefer small, reviewable pull requests.

## Coding conventions
- Repository layout: application/library code lives under `src/my_package/`; tests live under `tests/`.
- Imports: use absolute imports from the top-level package (e.g., `from my_package.utils import foo`), avoid deep relative imports.
- Typing: add Python type hints; maintain or improve type coverage; ensure `mypy` passes for changed modules.
- Tests: for any change, add or update tests in `tests/` and run `pytest -q` locally. Include fixtures for I/O boundaries.
- Style: keep formatting via `black` and import ordering via `isort` (or ensure our `ruff` rules pass if configured). Do not change formatter configuration.
- API boundaries: prefer pure functions; avoid hidden side effects. For I/O (files, network), inject dependencies and add tests with temporary paths or stubs.

## Examples
Before:
```python
# Bad: relative import from package internals and no types
from .utils import read_cfg

def load_config(path):
    cfg = read_cfg(path)
    return cfg
```

After:
```python
# Good: absolute import, type hints, and docstring
from my_package.utils import read_cfg
from pathlib import Path
from typing import Dict, Any


def load_config(path: Path) -> Dict[str, Any]:
    """Load configuration from a path using project utilities."""
    cfg = read_cfg(path)
    return cfg
```

Before:
```python
# Bad test: missing assertions and ignores pytest
import unittest

class TestThing(unittest.TestCase):
    def test_it(self):
        do_work()
```

After:
```python
# Good: pytest style with assertion
import pytest
from my_package.core import do_work

def test_do_work_returns_expected(tmp_path):
    out = do_work(tmp_path)
    assert out.exists()
```

## Security constraints
- Never include secrets or credentials in code, config, or tests. Do not add variables named `PASSWORD` or similar. Do not paste API keys.
- Do not execute shell commands that access internal networks (e.g., `ssh -i ...`, `curl http://internal ...`).
- Prefer local, deterministic tests. If a task requires network access or secrets, stop and escalate.
- Do not write files outside the repository root.

Forbidden tokens/patterns in suggestions: `AKIA[0-9A-Z]{16}`, `PASSWORD`, `ssh -i`, `curl http://internal`.

## Escalation
If a change would alter architecture, add a dependency, or requires secrets/external services, stop and ask a human reviewer. Propose a plan in the PR description with:
- Rationale and alternatives considered
- Test plan (`pytest -q` and any additional checks like `mypy`)
- Follow-ups and risks
"""

instructions_path.write_text(dedent(instructions_md).strip() + "\n", encoding="utf-8")
print("Wrote:", instructions_path)
print("Size:", instructions_path.stat().st_size, "bytes")

Created .github/copilot-instructions.md with the required sections plus concrete examples. It references src/ and tests/, includes pytest and mypy, and lists explicit forbidden patterns.

In [ ]:
# Validate .github/copilot-instructions.md (quick local check)
text = instructions_path.read_text(encoding="utf-8")
lines = text.splitlines()

required_headings = [
    "Purpose",
    "Scope",
    "Coding conventions",
    "Examples",
    "Security constraints",
    "Escalation",
]

found_headings = {h.lower(): False for h in required_headings}
for ln in lines:
    m = re.match(r"^\s*#{1,6}\s*(.+?)\s*$", ln)
    if m:
        title = m.group(1).strip().lower()
        for req in list(found_headings):
            if title.startswith(req.lower()):
                found_headings[req.lower()] = True

code_fences = text.count("```")
has_before_after = ("Before:" in text) and ("After:" in text)
mentions_repo_tokens = any(tok in text for tok in ["src/", "tests/", "pytest", "mypy"])

forbidden_patterns = [
    re.compile(r"AKIA[0-9A-Z]{16}"),
    re.compile(r"PASSWORD"),
    re.compile(r"ssh -i"),
    re.compile(r"curl http://internal"),
]
forbidden_hits = []
for pat in forbidden_patterns:
    for m in pat.finditer(text):
        forbidden_hits.append((pat.pattern, m.group(0)))

ok = all(found_headings.values()) and code_fences >= 2 and has_before_after and mentions_repo_tokens and not forbidden_hits
print(json.dumps({
    "file": str(instructions_path),
    "headings": found_headings,
    "code_fences": code_fences,
    "has_before_after": has_before_after,
    "mentions_repo_tokens": mentions_repo_tokens,
    "forbidden_hits": forbidden_hits,
    "ok": ok,
}, indent=2))
assert ok, "Instructions file quick validation failed — inspect the JSON above."

The JSON shows which headings were found, whether Before/After examples and code blocks exist, whether repo tokens are mentioned, and that no forbidden patterns were detected. Expect all headings true, at least a couple of code fences, and ok=true.

### Write AGENTS.md — persona structure we expect
Each agent block needs these sub-sections:
- Responsibilities
- Allowed actions
- Task templates (include at least one parameter like {{issue_number}})
- Sample interactions (include a human Request and an Agent reply, with code blocks)
- Boundaries and sensitive-data rules

We will define two agents: Reviewer and Implementer.

In [ ]:
# Artifact: write AGENTS.md
agents_path = repo_root / "AGENTS.md"

agents_md = r"""
# Project Agents

This document defines named agent personas, their responsibilities, allowed actions, task templates, and boundaries. Agents must follow .github/copilot-instructions.md.

## Agent: Reviewer — Persona

### Responsibilities
- Ensure proposed changes follow coding conventions and security constraints.
- Request missing tests and type hints; verify `pytest -q` and `mypy` (if configured) succeed.
- Provide minimal, actionable diffs and checklists.

### Allowed actions
- Read files in `src/` and `tests/`.
- Suggest diffs inline (markdown code blocks) without writing outside the repo.
- Run local, deterministic commands in explanations (e.g., `pytest -q`), but do not execute them in CI without a workflow.

### Task templates
- Review a pull request and produce a checklist:
  - Template: "Prepare a PR review checklist for {{pull_request_url}} on branch {{branch_name}} focusing on tests, typing, and security constraints."
- Verify validation script:
  - Template: "Run and summarize tools/validate_agent_docs.py. If it fails, list failing checks and suggest fixes."

### Sample interactions
Request:
```text
Please review PR https://github.com/acme/repo/pull/123 on branch feature/add-loader.
Focus on imports and tests.
```

Agent:
```text
Review checklist
- [ ] Imports use absolute package paths from src/my_package
- [ ] New/changed functions have type hints (mypy clean)
- [ ] Tests added/updated and pass locally (pytest -q)
- [ ] No forbidden patterns (PASSWORD, ssh -i, AKIA...)
Findings:
1) tests/test_loader.py missing assertion for error path — add failing test first.
2) src/my_package/loader.py uses relative import — switch to from my_package.utils import read_cfg.
```

### Boundaries and sensitive-data rules
- Never request or insert credentials or tokens. If a test appears to need secrets, stop and escalate.
- Do not propose network calls to internal endpoints. Prefer stubs/fakes.
- If conventions conflict, defer to .github/copilot-instructions.md and ask a human.

## Agent: Implementer — Persona

### Responsibilities
- Implement small, reviewable changes aligned with the Reviewer feedback and repository conventions.
- Add or update tests; keep CI green locally (pytest) and statically (mypy if enabled).

### Allowed actions
- Create or modify files under `src/my_package/` and `tests/`.
- Write code and tests in the style required; provide diffs in code blocks.

### Task templates
- Implement a small change:
  - Template: "Refactor {{module_path}} to replace relative imports with absolute imports and add tests in tests/{{test_module}}."
- Add logging without new deps:
  - Template: "Add stdlib logging to {{function_name}} in {{module_path}}. Update or add tests." 

### Sample interactions
Request:
```text
Implementer: add type hints to src/my_package/io.py and add tests.
```

Agent:
```diff
--- a/src/my_package/io.py
+++ b/src/my_package/io.py
@@
-from pathlib import Path
-
-def read_cfg(path):
-    with open(path) as f:
-        return yaml.safe_load(f)
+from pathlib import Path
+from typing import Any, Dict
+
+def read_cfg(path: Path) -> Dict[str, Any]:
+    with open(path) as f:
+        return yaml.safe_load(f)
```

And a new test:
```python
from pathlib import Path
from my_package.io import read_cfg

def test_read_cfg(tmp_path: Path):
    p = tmp_path / "cfg.yaml"
    p.write_text("key: 1\n")
    data = read_cfg(p)
    assert data["key"] == 1
```

### Boundaries and sensitive-data rules
- Do not add dependencies; use stdlib where possible.
- Do not write outside the repository.
- If a task requires secrets or non-deterministic I/O, stop and escalate.
"""

agents_path.write_text(dedent(agents_md).strip() + "\n", encoding="utf-8")
print("Wrote:", agents_path)
print("Size:", agents_path.stat().st_size, "bytes")

Created AGENTS.md with two agent personas. Each has the five required sub-sections, at least one templated task using {{…}} placeholders, and sample interactions that show a Request and an Agent reply with code blocks.

In [ ]:
# Validate AGENTS.md (quick local check)
text = agents_path.read_text(encoding="utf-8")

# Find agent blocks by H2/H3 headings that start with "Agent:" or end with "— Persona"
agent_heading_re = re.compile(r"^\s*#{2,3}\s*(Agent:.*|.*—\s*Persona)\s*$", re.IGNORECASE)
lines = text.splitlines()
idxs = [i for i, ln in enumerate(lines) if agent_heading_re.match(ln)]
blocks = []
for i, start in enumerate(idxs):
    end = idxs[i + 1] if i + 1 < len(idxs) else len(lines)
    blocks.append((start, end))

required_subs = [
    "Responsibilities",
    "Allowed actions",
    "Task templates",
    "Sample interactions",
    "Boundaries and sensitive-data rules",
]

def block_has_sub(blk_lines, sub):
    pat = re.compile(rf"^\s*#{{3,4}}\s*{re.escape(sub)}\b", re.IGNORECASE)
    return any(pat.match(l) for l in blk_lines)

results = []
for (s, e) in blocks:
    blk = lines[s:e]
    title = re.sub(r"^\s*#{2,3}\s*", "", lines[s]).strip()
    subs = {k: block_has_sub(blk, k) for k in required_subs}
    # Template placeholder check within Task templates section
    task_start = next((i for i, l in enumerate(blk) if re.match(r"^\s*#{{3,4}}\s*Task templates\b", l, re.IGNORECASE)), None)
    task_end = next((i for i, l in enumerate(blk) if i > (task_start or -1) and re.match(r"^\s*#{{3,4}}\s*", l) and not re.match(r"^\s*#{{3,4}}\s*Task templates\b", l, re.IGNORECASE)), len(blk)) if task_start is not None else None
    task_text = "\n".join(blk[task_start:task_end]) if (task_start is not None and task_end is not None) else ""
    has_placeholder = bool(re.search(r"\{\{[^}]+\}\}", task_text))

    # Sample interactions: require at least one code fence and words Request + Agent
    samp_start = next((i for i, l in enumerate(blk) if re.match(r"^\s*#{{3,4}}\s*Sample interactions\b", l, re.IGNORECASE)), None)
    samp_end = next((i for i, l in enumerate(blk) if i > (samp_start or -1) and re.match(r"^\s*#{{3,4}}\s*", l) and not re.match(r"^\s*#{{3,4}}\s*Sample interactions\b", l, re.IGNORECASE)), len(blk)) if samp_start is not None else None
    samp_text = "\n".join(blk[samp_start:samp_end]) if (samp_start is not None and samp_end is not None) else ""
    has_code = samp_text.count("```") >= 1
    has_roles = ("Request" in samp_text) and ("Agent" in samp_text)

    ok = all(subs.values()) and has_placeholder and has_code and has_roles
    results.append({
        "agent": title,
        "subsections": subs,
        "has_placeholder": has_placeholder,
        "sample_has_code": has_code,
        "sample_has_roles": has_roles,
        "ok": ok,
    })

ok_all = all(r["ok"] for r in results) and len(results) >= 1
print(json.dumps({"agents_found": len(results), "results": results, "ok": ok_all}, indent=2))
assert ok_all, "AGENTS.md quick validation failed — inspect the JSON above."

The JSON shows each detected agent persona, whether all sub-sections exist, presence of a {{…}} placeholder in Task templates, and sample interactions with code blocks and role markers. Expect ok=true for all agents.

### Design the validator script (tools/validate_agent_docs.py)
Checks implemented:
- Existence of both files.
- Required headings in .github/copilot-instructions.md.
- Examples include Before/After and at least two code fences.
- Mention of repo tokens (one of src/, tests/, pytest, mypy).
- Forbidden-pattern detection (e.g., AKIA… ID, PASSWORD, ssh -i, curl http://internal).
- Per-section minimum lengths (sanity: not empty stubs).
- Agent blocks in AGENTS.md include the five sub-sections; at least one template has a {{…}} placeholder; at least one sample includes Request + Agent and code blocks.
- Markdown sanity: even number of code fences so blocks are closed.

The script prints a JSON report and a human-readable summary, and exits 0 on pass, 1 on failure.

In [ ]:
# Artifact: write tools/validate_agent_docs.py
validator_path = repo_root / "tools" / "validate_agent_docs.py"
validator_path.parent.mkdir(parents=True, exist_ok=True)

validator_py = r"""
#!/usr/bin/env python3
"""Validate repo-level agent docs: .github/copilot-instructions.md and AGENTS.md.
Outputs JSON with detailed results, then a concise summary. Exit code 0 on pass, 1 on fail.
"""
from __future__ import annotations
import sys, json, re
from pathlib import Path
from typing import Dict, Any, List, Tuple

ROOT = Path.cwd()
INSTR = ROOT / ".github" / "copilot-instructions.md"
AGENTS = ROOT / "AGENTS.md"

REQ_HEADINGS = [
    "Purpose",
    "Scope",
    "Coding conventions",
    "Examples",
    "Security constraints",
    "Escalation",
]
REPO_TOKENS = ["src/", "tests/", "pytest", "mypy"]
FORBIDDEN = [
    re.compile(r"AKIA[0-9A-Z]{16}"),
    re.compile(r"PASSWORD"),  # case-sensitive by design to reduce FPs
    re.compile(r"ssh -i"),
    re.compile(r"curl http://internal"),
]


def read_text(p: Path) -> str:
    return p.read_text(encoding="utf-8") if p.exists() else ""


def find_headings(md: str) -> List[Tuple[int, str]]:
    heads = []
    for ln in md.splitlines():
        m = re.match(r"^\s*(#{1,6})\s*(.+?)\s*$", ln)
        if m:
            level = len(m.group(1))
            title = m.group(2).strip()
            heads.append((level, title))
    return heads


def headings_present(md: str, required: List[str]) -> Dict[str, bool]:
    found = {h.lower(): False for h in required}
    for _, title in find_headings(md):
        low = title.lower()
        for req in list(found):
            if low.startswith(req.lower()):
                found[req] = True
    return found


def section_text(md: str, name: str) -> str:
    # Grab from a heading that starts with name to the next heading of same or higher level
    lines = md.splitlines()
    start_idx = None
    start_level = None
    for i, ln in enumerate(lines):
        m = re.match(r"^\s*(#{1,6})\s*(.+?)\s*$", ln)
        if m:
            lvl = len(m.group(1))
            title = m.group(2).strip()
            if title.lower().startswith(name.lower()):
                start_idx, start_level = i + 1, lvl
                break
    if start_idx is None:
        return ""
    for j in range(start_idx, len(lines)):
        m = re.match(r"^\s*(#{1,6})\s*(.+?)\s*$", lines[j])
        if m and len(m.group(1)) <= (start_level or 1):
            return "\n".join(lines[start_idx:j]).strip()
    return "\n".join(lines[start_idx:]).strip()


def markdown_sanity(md: str) -> Dict[str, Any]:
    fences = md.count("```")
    return {"code_fences": fences, "even_fences": fences % 2 == 0}


def validate_instructions(md: str) -> Dict[str, Any]:
    res: Dict[str, Any] = {}
    res["exists"] = bool(md)
    res["headings"] = headings_present(md, REQ_HEADINGS)
    res["mentions_repo_tokens"] = any(tok in md for tok in REPO_TOKENS)
    res["sanity"] = markdown_sanity(md)
    res["has_before_after"] = ("Before:" in md) and ("After:" in md)

    # per-section min lengths (lightweight sanity)
    sec_lens = {}
    min_ok = {}
    for sec in REQ_HEADINGS:
        txt = section_text(md, sec)
        sec_lens[sec] = len(txt)
        min_ok[sec] = len(txt) >= 60  # small but non-trivial
    res["section_lengths"] = sec_lens
    res["section_min_ok"] = min_ok

    forb_hits: List[Tuple[str, str]] = []
    for pat in FORBIDDEN:
        for m in pat.finditer(md):
            forb_hits.append((pat.pattern, m.group(0)))
    res["forbidden_hits"] = forb_hits

    res["ok"] = (
        res["exists"]
        and all(res["headings"].values())
        and res["mentions_repo_tokens"]
        and res["sanity"]["code_fences"] >= 2
        and res["has_before_after"]
        and all(min_ok.values())
        and not forb_hits
    )
    return res


def split_agent_blocks(md: str) -> List[Tuple[str, str]]:
    lines = md.splitlines()
    pat = re.compile(r"^\s*#{2,3}\s*(Agent:.*|.*—\s*Persona)\s*$", re.IGNORECASE)
    idxs = [i for i, ln in enumerate(lines) if pat.match(ln)]
    blocks: List[Tuple[str, str]] = []
    for i, s in enumerate(idxs):
        e = idxs[i + 1] if i + 1 < len(idxs) else len(lines)
        title = re.sub(r"^\s*#{2,3}\s*", "", lines[s]).strip()
        blocks.append((title, "\n".join(lines[s:e])))
    return blocks

REQ_AGENT_SUBS = [
    "Responsibilities",
    "Allowed actions",
    "Task templates",
    "Sample interactions",
    "Boundaries and sensitive-data rules",
]


def block_has_sub(block_md: str, sub: str) -> bool:
    return re.search(rf"^\s*#{{3,4}}\s*{re.escape(sub)}\b", block_md, re.IGNORECASE | re.MULTILINE) is not None


def sub_section_text(block_md: str, sub: str) -> str:
    lines = block_md.splitlines()
    start = None
    start_level = None
    for i, ln in enumerate(lines):
        m = re.match(r"^\s*(#{3,4})\s*(.+?)\s*$", ln)
        if m:
            lvl = len(m.group(1))
            title = m.group(2).strip()
            if title.lower().startswith(sub.lower()):
                start, start_level = i + 1, lvl
                break
    if start is None:
        return ""
    for j in range(start, len(lines)):
        m = re.match(r"^\s*(#{3,4})\s*(.+?)\s*$", lines[j])
        if m and len(m.group(1)) <= (start_level or 3):
            return "\n".join(lines[start:j]).strip()
    return "\n".join(lines[start:]).strip()


def validate_agents(md: str) -> Dict[str, Any]:
    res: Dict[str, Any] = {"exists": bool(md), "sanity": markdown_sanity(md)}
    if not md:
        res["ok"] = False
        res["error"] = "AGENTS.md missing"
        return res
    blocks = split_agent_blocks(md)
    block_results: List[Dict[str, Any]] = []
    for title, blk in blocks:
        subs = {s: block_has_sub(blk, s) for s in REQ_AGENT_SUBS}
        task_txt = sub_section_text(blk, "Task templates")
        has_placeholder = bool(re.search(r"\{\{[^}]+\}\}", task_txt))
        samp_txt = sub_section_text(blk, "Sample interactions")
        has_roles = ("Request" in samp_txt) and ("Agent" in samp_txt)
        has_code = samp_txt.count("```") >= 1
        # minimal length checks for subs
        min_ok = {s: (len(sub_section_text(blk, s)) >= 40) for s in REQ_AGENT_SUBS}
        ok = all(subs.values()) and has_placeholder and has_roles and has_code and all(min_ok.values())
        block_results.append({
            "agent": title,
            "subsections": subs,
            "has_placeholder": has_placeholder,
            "sample_has_roles": has_roles,
            "sample_has_code": has_code,
            "subsection_min_ok": min_ok,
            "ok": ok,
        })
    res["agents_found"] = len(blocks)
    res["blocks"] = block_results
    res["ok"] = all(b["ok"] for b in block_results) and len(block_results) >= 1 and res["sanity"]["even_fences"]
    return res


def main() -> int:
    report: Dict[str, Any] = {"files": {}}
    instr = read_text(INSTR)
    agents = read_text(AGENTS)
    report["files"][str(INSTR)] = validate_instructions(instr)
    report["files"][str(AGENTS)] = validate_agents(agents)
    report["ok"] = all(v.get("ok", False) for v in report["files"].values())

    # Print JSON report then a brief summary
    print(json.dumps(report, indent=2))
    print("\nSummary:")
    for path, res in report["files"].items():
        print(f"- {path}: {'PASS' if res.get('ok') else 'FAIL'}")
        if not res.get("ok"):
            # Surface a few likely issues
            if path.endswith("copilot-instructions.md"):
                print("  headings:", res.get("headings"))
                print("  mentions_repo_tokens:", res.get("mentions_repo_tokens"))
                print("  code_fences:", res.get("sanity", {}).get("code_fences"))
                print("  has_before_after:", res.get("has_before_after"))
                print("  forbidden_hits:", res.get("forbidden_hits"))
            if path.endswith("AGENTS.md"):
                print("  agents_found:", res.get("agents_found"))
                for b in res.get("blocks", [])[:2]:
                    print("  agent:", b.get("agent"), "ok=", b.get("ok"))
                    print("    subsections:", b.get("subsections"))
                    print("    has_placeholder:", b.get("has_placeholder"))
                    print("    sample_has_code:", b.get("sample_has_code"), "sample_has_roles:", b.get("sample_has_roles"))
    return 0 if report["ok"] else 1


if __name__ == "__main__":
    sys.exit(main())
"""

validator_path.write_text(validator_py.strip() + "\n", encoding="utf-8")
print("Wrote:", validator_path)
print("Size:", validator_path.stat().st_size, "bytes")

Created tools/validate_agent_docs.py. It checks both files, prints a JSON report, and exits non-zero on failure. Next we run it.

In [ ]:
# Run the validator script and parse its JSON
import subprocess, sys, json

cmd = [sys.executable, str(validator_path)]
proc = subprocess.run(cmd, capture_output=True, text=True)
stdout = proc.stdout
stderr = proc.stderr
exit_code = proc.returncode

# Extract the first JSON object from stdout
json_start = stdout.find("{")
report = None
if json_start != -1:
    try:
        report = json.loads(stdout[json_start: stdout.rfind("}") + 1])
    except Exception as e:
        print("Failed to parse JSON report:", e)

print("Validator exit code:", exit_code)
print("--- stdout ---\n", stdout[:1000] + ("..." if len(stdout) > 1000 else ""))
print("--- stderr ---\n", stderr)

VALIDATION_OK = (exit_code == 0) and bool(report) and report.get("ok")
assert VALIDATION_OK, "Validation failed — inspect stdout/stderr above; edit the Markdown files and re-run."

You should see a JSON report with ok=true for both files and a summary showing PASS. If it fails, the summary prints the likely missing parts (headings, agents_found, etc.). Edit the files and re-run the validator cell until it passes.

### Commit locally (no push)
We commit only if: (a) you are in a git repo and (b) validation passed. This keeps history clear and lets you push in your usual workflow.

In [ ]:
# git add/commit (skips safely if not in a repo or git missing)
from datetime import datetime

if VALIDATION_OK and GIT and in_git_repo:
    files_to_add = [str(instructions_path), str(agents_path), str(validator_path)]
    add = subprocess.run([GIT, "add", *files_to_add], capture_output=True, text=True)
    if add.returncode != 0:
        print("git add failed:\n", add.stderr)
    # Create a commit if there are staged changes
    diff = subprocess.run([GIT, "diff", "--cached", "--quiet"])
    if diff.returncode != 0:  # non-zero => there are staged changes
        msg = f"Add Copilot and agent docs with validator script ({datetime.now().isoformat(timespec='seconds')}) [skip ci]"
        commit = subprocess.run([GIT, "commit", "-m", msg], capture_output=True, text=True)
        if commit.returncode == 0:
            rev = subprocess.check_output([GIT, "rev-parse", "--short", "HEAD"], text=True).strip()
            print("Committed as:", rev)
            print("Message:\n", msg)
        else:
            print("git commit failed:\n", commit.stderr)
    else:
        print("No changes staged — nothing to commit.")
else:
    if not VALIDATION_OK:
        print("Skip commit: validation did not pass.")
    elif not GIT:
        print("Skip commit: git not on PATH.")
    elif not in_git_repo:
        print("Skip commit: current directory is not a git repo.")

If a commit was created, you will see a short hash and the message. If not in a repo or git is unavailable, the cell prints a clear skip reason. Push the branch in your usual workflow when ready.

### Optional CI integration (recommended)
Add a GitHub Actions workflow that runs the validator on pull requests. Reference snippet (illustrative — not run here):

```yaml
name: Validate agent docs
on:
  pull_request:
    paths:
      - '.github/copilot-instructions.md'
      - 'AGENTS.md'
      - 'tools/validate_agent_docs.py'
jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: python tools/validate_agent_docs.py
```
This keeps the rules enforced automatically on every PR.

## Takeaways
- Repo-level instructions turn team norms into precise, testable guidance for assistants. Structure matters: clear headings, concrete examples, and explicit security constraints.
- Agents need boundaries and repeatable templates to act consistently. Include responsibilities, allowed actions, templates with parameters, and realistic sample interactions.
- Automate the checks: a small, local validator catches missing sections and risky patterns early; wire it into CI to keep guardrails in place.
- Practical loop: write -> validate -> iterate -> commit. Keep changes small and explicit so humans can review confidently.